In [6]:
import pandas as pd
import numpy as np

In [7]:
static      = pd.read_csv("../data/raw/pmemo/annotations/static_annotations.csv")
static_std  = pd.read_csv("../data/raw/pmemo/annotations/static_annotations_std.csv")
dynamic     = pd.read_csv("../data/raw/pmemo/annotations/dynamic_annotations.csv")

print(static.shape)
print(static.head())

(767, 3)
   musicId  Arousal(mean)  Valence(mean)
0        1         0.4000         0.5750
1        4         0.2625         0.2875
2        5         0.1500         0.2000
3        6         0.5125         0.3500
4        7         0.7000         0.7250


In [8]:
merged = static.merge(static_std, on="musicId")

before = len(merged)
merged = merged[merged["Valence(std)"] <= 0.25].copy()
print(f"Kept {len(merged)} / {before} tracks")

Kept 736 / 767 tracks


In [9]:
merged["valence_norm"] = ((merged["Valence(mean)"] - 0.5) * 2).clip(-0.99, 0.99)
merged["arousal_norm"] = ((merged["Arousal(mean)"] - 0.5) * 2).clip(-0.99, 0.99)
merged["valence_01"]   = merged["Valence(mean)"]   # keep original for buckets

print(merged[["musicId","Valence(mean)","valence_norm"]].head())

   musicId  Valence(mean)  valence_norm
0        1         0.5750         0.150
1        4         0.2875        -0.425
2        5         0.2000        -0.600
3        6         0.3500        -0.300
4        7         0.7250         0.450


In [10]:
def get_action_bucket(valence, energy, tempo):
    v = 0 if valence < 0.33 else 1
    e = 0 if energy  < 0.4  else 1
    t = 0 if tempo   < 100  else 1
    return int(v * 4 + e * 2 + t)

# No features.csv yet — use neutral defaults
# If you have features.csv, load it and merge energy/tempo before this step
merged["energy"] = 0.5
merged["tempo"]  = 120.0

merged["action_bucket"] = merged.apply(
    lambda r: get_action_bucket(r["valence_01"], r["energy"], r["tempo"]),
    axis=1
)

print(merged["action_bucket"].value_counts().sort_index())

action_bucket
3     53
7    683
Name: count, dtype: int64


In [11]:
out = merged[["musicId","valence_01","valence_norm","arousal_norm",
              "action_bucket","energy","tempo"]].rename(columns={"musicId":"song_id"})

out.to_csv("../data/processed/music_pmemo_clean.csv", index=False)
print(f"Saved {len(out)} tracks")
print(out.head())

Saved 736 tracks
   song_id  valence_01  valence_norm  arousal_norm  action_bucket  energy  \
0        1      0.5750         0.150        -0.200              7     0.5   
1        4      0.2875        -0.425        -0.475              3     0.5   
2        5      0.2000        -0.600        -0.700              3     0.5   
3        6      0.3500        -0.300         0.025              7     0.5   
4        7      0.7250         0.450         0.400              7     0.5   

   tempo  
0  120.0  
1  120.0  
2  120.0  
3  120.0  
4  120.0  
